# Exercícios — Regressão e métricas

Resolva no papel antes de abrir cada solução. As células de solução vêm recolhidas (`# @title`).

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## Exercício 1 — RMSE e R² na mão e no sklearn

Regressão linear de `bmi` no `diabetes`; RMSE e R² no teste, à mão e com o scikit-learn.

In [ ]:
# @title Solução
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

d = load_diabetes()
X = d.data[:, [2]]   # bmi
y = d.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=SEMENTE)
modelo = LinearRegression().fit(X_tr, y_tr)
previsto = modelo.predict(X_te)
# na mao
rmse_mao = np.sqrt(np.mean((y_te - previsto) ** 2))
r2_mao = 1 - np.sum((y_te - previsto)**2) / np.sum((y_te - y_te.mean())**2)
print("RMSE (mao):", round(rmse_mao, 2), "| sklearn:", round(np.sqrt(mean_squared_error(y_te, previsto)), 2))
print("R2   (mao):", round(r2_mao, 3), "| sklearn:", round(r2_score(y_te, previsto), 3))

## Exercício 2 — Escolher o grau do polinômio

In [ ]:
# @title Solução
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

x = np.linspace(0, 1, 60)
y = np.sin(2*np.pi*x) + np.random.normal(0, 0.25, size=x.shape)
xtr, xte, ytr, yte = train_test_split(x, y, test_size=0.4, random_state=SEMENTE)
melhor, melhor_erro = None, 1e9
for grau in range(1, 16):
    m = make_pipeline(PolynomialFeatures(grau), LinearRegression()).fit(xtr.reshape(-1,1), ytr)
    erro = mean_squared_error(yte, m.predict(xte.reshape(-1,1)))
    if erro < melhor_erro: melhor, melhor_erro = grau, erro
print("grau que minimiza o erro de validacao:", melhor, "| MSE:", round(melhor_erro, 3))

## Exercício 3 — Ridge, Lasso e esparsidade

In [ ]:
# @title Solução
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso

X = d.data; y = d.target
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X, y)
lasso = make_pipeline(StandardScaler(), Lasso(alpha=1.0)).fit(X, y)
nz_ridge = int(np.sum(np.abs(ridge.named_steps["ridge"].coef_) > 1e-6))
nz_lasso = int(np.sum(np.abs(lasso.named_steps["lasso"].coef_) > 1e-6))
print("coeficientes != 0 -> Ridge:", nz_ridge, "de 10 | Lasso:", nz_lasso, "de 10")
print("a Ridge nao zera nenhum; o Lasso zera varios (seleciona variaveis).")

## Exercício 4 — Interpretar um coeficiente

In [ ]:
# @title Solução
modelo = make_pipeline(StandardScaler(), LinearRegression()).fit(X, y)
coefs = modelo.named_steps["linearregression"].coef_
j = int(np.argmax(coefs))
print("maior coeficiente positivo:", d.feature_names[j], "=", round(coefs[j], 1))
print("Leitura: +1 desvio-padrao em", d.feature_names[j],
      "-> +", round(coefs[j], 1), "na progressao prevista, mantidos os demais fixos.")